In [2]:
import json
import random
import re
import unicodedata
from huggingface_hub import login
from typing import Dict, List, Tuple

import regex
import numpy as np
from datasets import Dataset, DatasetDict, load_dataset
from tqdm.auto import tqdm

try:  # aksharamukha is only needed for the Sanskrit (Devanagari->Telugu) path
    from aksharamukha import transliterate as _aksharamukha
except Exception:  # keep the module importable without it (English/Telugu still work)
    _aksharamukha = None

/Users/xai/Personal/Projects/TeluguOCR/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# --------------------------------------------------------------------------------------
# Language-specific text cleaning (applied as the FIRST map per corpus, before tokenizing)
# --------------------------------------------------------------------------------------
# English: normalize smart punctuation, strip accents/combining marks, force pure ASCII.
PUNCT_MAP = str.maketrans({
    "\u201c": '"', "\u201d": '"',      # “ ”
    "\u2018": "'", "\u2019": "'",      # ‘ ’
    "\u2014": "-", "\u2013": "-",      # — –
    "\u2026": "...",                    # …
})


def clean_english(example, text_col="text", **_):
    text = example[text_col]
    text = text.translate(PUNCT_MAP)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    example[text_col] = text.encode("ascii", "ignore").decode("ascii")
    return example


# Telugu / Sanskrit share this whitelist: tab/newline/CR, printable ASCII (U+20..U+7E),
# the Telugu block (U+0C00..U+0C7F), danda/double-danda (U+0964/U+0965), Vedic tone U+1CDA.
_INDIC_FILTER = re.compile(r"[^\t\n\r -~\u0C00-\u0C7F\u0964\u0965\u1CDA]")


def clean_telugu_text(example, text_col="text", **_):
    text = example[text_col]
    text = _INDIC_FILTER.sub("", text)
    text = unicodedata.normalize("NFKC", text)
    example[text_col] = text
    return example


def transliterate_sanskrit(example, text_col="text", src_script="Devanagari", **_):
    """Transliterate source script -> Telugu (aksharamukha), preserving dandas."""
    if _aksharamukha is None:
        raise ImportError("aksharamukha is required for the Sanskrit path: "
                          "pip install aksharamukha")
    text = example[text_col]
    text = text.replace("\u0964", "__DANDA__").replace("\u0965", "__DDANDA__")
    text = _aksharamukha.process(src_script, "Telugu", text)
    text = text.replace("__DANDA__", "\u0964").replace("__DDANDA__", "\u0965")
    text = _INDIC_FILTER.sub("", text)
    text = unicodedata.normalize("NFKC", text)
    example[text_col] = text
    return example


# lang_code -> cleaner applied before tokenization
CLEANERS = {
    "english": clean_english,
    "telugu": clean_telugu_text,
    "sanskrit": transliterate_sanskrit,
}


# --------------------------------------------------------------------------------------
# Grapheme helpers (Unicode extended grapheme clusters via regex \X)
# --------------------------------------------------------------------------------------
_GRAPHEME_RE = regex.compile(r"\X")


def graphemes(s: str) -> List[str]:
    return _GRAPHEME_RE.findall(s)


def glen(s: str) -> int:
    return len(_GRAPHEME_RE.findall(s))


# --------------------------------------------------------------------------------------
# Exact apportionment (largest-remainder) so totals hit the requested number exactly
# --------------------------------------------------------------------------------------
def apportion(total: int, weights: List[float]) -> List[int]:
    """Split `total` into len(weights) integers that sum to exactly `total`,
    proportional to weights, using the largest-remainder method."""
    s = float(sum(weights))
    raw = [total * w / s for w in weights]
    floors = [int(x) for x in raw]
    remainder = total - sum(floors)
    order = sorted(range(len(raw)), key=lambda i: raw[i] - floors[i], reverse=True)
    for i in range(remainder):
        floors[order[i]] += 1
    return floors


def bucket_lengths(rng: str) -> List[int]:
    lo, hi = map(int, rng.split("-"))
    return list(range(lo, hi + 1))


# --------------------------------------------------------------------------------------
# Corpus preprocessing -> per-article words/glens + short-line sampling pools
# --------------------------------------------------------------------------------------
SHORT_MAX = 3          # target lengths 1..SHORT_MAX use the short-line path
LATIN_DIGITS = list("0123456789")
TELUGU_DIGITS = list("\u0c66\u0c67\u0c68\u0c69\u0c6a\u0c6b\u0c6c\u0c6d\u0c6e\u0c6f")
PUNCT = list(".,!?;:-()\"'/%")                 # ASCII only (survives every cleaner)
DANDA = ["\u0964", "\u0965"]                   # । ॥  (kept by the Indic filter)
LATIN_LETTERS = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ")


def _tokenize_batch(batch, text_col):
    words_col, glens_col = [], []
    for t in batch[text_col]:
        w = t.split() if t else []
        words_col.append(w)
        glens_col.append([glen(x) for x in w])
    return {"words": words_col, "word_glens": glens_col}


def preprocess_corpus(
    ds: Dataset,
    text_col: str,
    lang_code: str,
    num_proc: int = 1,
    long_pool_cap: int = 200_000,
    exact_pool_cap: int = 50_000,
    seed: int = 0,
    clean: bool = True,
    sanskrit_src: str = "Devanagari",
    sanskrit_transliterate: bool = False,
    max_articles: int | None = None,
) -> Dict:
    """Return a corpus object: an Arrow-backed (memory-mapped) article table plus a
    compact capacity index and capped short-line pools. The full text is never pulled
    into Python objects, so peak RAM stays bounded regardless of corpus size."""
    # 0) optional subsample (useful for quick Kaggle test runs on huge corpora)
    if max_articles is not None and len(ds) > max_articles:
        ds = ds.shuffle(seed=seed).select(range(max_articles))

    # 1) language-specific cleaning (first pass).
    #    Sanskrit: if the corpus is ALREADY transcribed to Telugu script (the default now),
    #    just apply the Telugu filter+NFKC; only run aksharamukha when it is still Devanagari.
    clean_fn = None
    fn_kwargs = {"text_col": text_col}
    if clean:
        if lang_code == "sanskrit":
            if sanskrit_transliterate:
                clean_fn = transliterate_sanskrit
                fn_kwargs["src_script"] = sanskrit_src
            else:
                clean_fn = clean_telugu_text        # already Telugu script
        else:
            clean_fn = CLEANERS.get(lang_code)
    if clean_fn is not None:
        ds = ds.map(clean_fn, num_proc=num_proc, fn_kwargs=fn_kwargs,
                    desc=f"clean[{lang_code}]")

    # 2) tokenize + per-word grapheme lengths
    ds = ds.map(
        _tokenize_batch,
        batched=True,
        num_proc=num_proc,
        fn_kwargs={"text_col": text_col},
        remove_columns=ds.column_names,
        desc=f"tokenize[{lang_code}]",
    )

    rnd = random.Random(seed)

    # 3) SINGLE STREAMING PASS over the tokenized corpus.
    # We never pull the whole `words` column into Python (that is what OOM-killed the
    # kernel). Articles stay in the memory-mapped Arrow dataset and are indexed on demand
    # during generation. Here we only build: a compact int capacity array + capped pools.
    n = len(ds)
    caps = np.zeros(n, dtype=np.int64)   # per-article grapheme capacity; 0 == unusable
    exact: Dict[int, set] = {L: set() for L in range(1, SHORT_MAX + 1)}
    long_tokens: List[str] = []

    i = 0
    _bs = 2000
    _nb = (n + _bs - 1) // _bs
    for batch in tqdm(ds.iter(batch_size=_bs), total=_nb,
                      desc=f"index[{lang_code}]", unit="batch"):
        for words, wl in zip(batch["words"], batch["word_glens"]):
            if not words:
                i += 1
                continue
            caps[i] = sum(wl) + (len(words) - 1)  # +1 grapheme per inter-word space
            i += 1
            for w, L in zip(words, wl):
                if 1 <= L <= SHORT_MAX:
                    if len(exact[L]) < exact_pool_cap:
                        exact[L].add(w)
                elif L > SHORT_MAX:
                    if len(long_tokens) < long_pool_cap:
                        long_tokens.append(w)
                    elif rnd.random() < 0.02:  # light reservoir refresh past the cap
                        long_tokens[rnd.randrange(long_pool_cap)] = w
            # akshara coverage for length-1: harvest single graphemes seen inside words
            if len(exact[1]) < exact_pool_cap and rnd.random() < 0.05:
                for g in graphemes(words[rnd.randrange(len(words))]):
                    if len(exact[1]) >= exact_pool_cap:
                        break
                    exact[1].add(g)

    # ---- language-aware injection of digits / numbers / punctuation / letters ----
    for ch in LATIN_DIGITS + PUNCT:
        exact[1].add(ch)
    if lang_code in ("telugu", "sanskrit"):
        for ch in TELUGU_DIGITS + DANDA:
            exact[1].add(ch)
    if lang_code == "english":
        for ch in LATIN_LETTERS:
            exact[1].add(ch)
    for L in range(2, SHORT_MAX + 1):
        for _ in range(200):  # numeric strings of this grapheme length
            exact[L].add("".join(rnd.choice(LATIN_DIGITS) for _ in range(L)))
            if lang_code in ("telugu", "sanskrit"):
                exact[L].add("".join(rnd.choice(TELUGU_DIGITS) for _ in range(L)))

    exact_lists = {L: list(v) for L, v in exact.items()}

    # capacity index for fast eligibility (articles with capacity >= target).
    # numpy argsort + searchsorted keeps this O(log n) and out of Python-object memory.
    order = np.argsort(caps, kind="stable")
    sorted_caps = caps[order]

    return {
        "lang_code": lang_code,
        "articles_ds": ds,             # memory-mapped Arrow; indexed by row at gen time
        "n": n,
        "caps": caps,                  # np.int64[n]
        "order": order,                # np.int64[n] article indices sorted by capacity asc
        "sorted_caps": sorted_caps,    # np.int64[n] parallel sorted capacities
        "exact": exact_lists,          # {1: [...], 2: [...], 3: [...]}
        "long": long_tokens,           # tokens with glen > SHORT_MAX (capped)
        "max_cap": int(sorted_caps[-1]) if n else 0,
    }


# --------------------------------------------------------------------------------------
# Line construction primitives
# --------------------------------------------------------------------------------------
def _get_article(corp: Dict, idx: int):
    """Fetch one article's (words, word_glens) from the memory-mapped Arrow dataset."""
    row = corp["articles_ds"][int(idx)]
    return row["words"], row["word_glens"]


def _pick_eligible_article(corp: Dict, target: int, rnd: random.Random) -> int:
    """Return an article index whose capacity >= target, or -1 if none."""
    sc = corp["sorted_caps"]
    n = corp["n"]
    pos = int(np.searchsorted(sc, target, side="left"))
    if pos >= n:
        return -1
    j = rnd.randrange(pos, n)
    return int(corp["order"][j])


def _append_from_article(words, wl, target, rnd) -> str | None:
    """Append whole words from a random start until length crosses `target`;
    keep-or-drop the crossing word 50/50. Return None if it never reaches target."""
    n = len(words)
    start = rnd.randrange(n)
    cur, cur_len, crossed = "", 0, False
    i = start
    while i < n:
        add = wl[i] + (1 if cur else 0)  # +1 grapheme for the separating space
        if cur_len + add >= target:
            if cur == "" or rnd.random() < 0.5:  # keep the crossing word
                cur = (cur + " " + words[i]) if cur else words[i]
            crossed = True
            break
        cur = (cur + " " + words[i]) if cur else words[i]
        cur_len += add
        i += 1
    return cur if crossed else None


def _overlength_from_article(words, wl, target, rnd) -> str:
    """Build a chunk from a random *feasible* start with grapheme length >= target.
    A start is feasible if its suffix capacity can reach `target`; else fall back to 0."""
    n = len(words)
    start = 0
    for _ in range(4):
        s = rnd.randrange(n)
        if sum(wl[s:]) + (n - 1 - s) >= target:
            start = s
            break
    cur, cur_len = "", 0
    i = start
    while i < n and cur_len < target:
        cur = (cur + " " + words[i]) if cur else words[i]
        cur_len += wl[i] + (1 if i > start else 0)
        i += 1
    return cur


def _append_line(corp, target, rnd, max_tries=8) -> Tuple[str, bool, str]:
    for _ in range(max_tries):
        idx = _pick_eligible_article(corp, target, rnd)
        if idx < 0:
            break
        words, wl = _get_article(corp, idx)
        out = _append_from_article(words, wl, target, rnd)
        if out:
            return out, False, "none"
    # fallback: force an exact end-cut so we always return something
    return _abrupt_line(corp, target, "end", rnd)


def _abrupt_line(corp, target, side, rnd) -> Tuple[str, bool, str]:
    idx = _pick_eligible_article(corp, target, rnd)
    if idx < 0:  # corpus too small for this target: use the largest article we have
        idx = int(corp["order"][-1]) if corp["n"] else -1
    if idx < 0:
        return _degrade_atom(corp, target, rnd)
    words, wl = _get_article(corp, idx)
    chunk = _overlength_from_article(words, wl, target, rnd)
    g = graphemes(chunk)
    if not g:
        return _degrade_atom(corp, target, rnd)
    if len(g) < target:            # article genuinely shorter than target: return all we have
        return chunk, True, side if side in ("begin", "end") else "end"
    if side == "end":              # line ENDS mid-word: keep first `target` graphemes
        return "".join(g[:target]), True, "end"
    return "".join(g[-target:]), True, "begin"   # line STARTS mid-word: keep last `target`


def _degrade_atom(corp, target, rnd) -> Tuple[str, bool, str]:
    """Absolute last resort when a corpus cannot supply `target` graphemes."""
    for L in (target, SHORT_MAX, 2, 1):
        pool = corp["exact"].get(L)
        if pool:
            return rnd.choice(pool), False, "none"
    return rnd.choice(LATIN_DIGITS), False, "none"


def _short_frag(corp, target, rnd) -> Tuple[str, bool, str]:
    """A fragment cut from a token longer than `target` (for target 1..3)."""
    pool = corp["long"] or corp["exact"].get(SHORT_MAX) or corp["exact"][1]
    for _ in range(6):
        tok = rnd.choice(pool)
        g = graphemes(tok)
        if len(g) > target:
            if rnd.random() < 0.5:
                return "".join(g[:target]), True, "end"     # word chopped at its end
            return "".join(g[-target:]), True, "begin"      # word chopped at its start
    # last resort: exact-length atom
    ex = corp["exact"].get(target)
    return (rnd.choice(ex) if ex else rnd.choice(LATIN_DIGITS)), False, "none"


def _short_line(corp, target, rnd, frag_ratio) -> Tuple[str, bool, str]:
    if rnd.random() < frag_ratio:
        return _short_frag(corp, target, rnd)
    ex = corp["exact"].get(target)
    if ex:
        return rnd.choice(ex), False, "none"
    return _short_frag(corp, target, rnd)


def generate_one(corp, target, rnd, abrupt_per, frag_ratio) -> Tuple[str, bool, str]:
    if target <= SHORT_MAX:
        return _short_line(corp, target, rnd, frag_ratio)
    r = rnd.random()
    if r < abrupt_per / 2.0:
        return _abrupt_line(corp, target, "end", rnd)
    if r < abrupt_per:
        return _abrupt_line(corp, target, "begin", rnd)
    return _append_line(corp, target, rnd)


# --------------------------------------------------------------------------------------
# Plan construction (exact counts) and parallel generation
# --------------------------------------------------------------------------------------
def make_plan(total: int, lang_pcts: Dict[str, float], length_dist: Dict[str, float],
              text_type_pcts: Dict[str, float] | None = None,
              random_langs: Tuple[str, ...] = (),
              seed: int = 0) -> Dataset:
    langs = list(lang_pcts)
    lang_counts = apportion(total, [lang_pcts[l] for l in langs])
    ranges = list(length_dist)

    # Natural/random split (fractions must sum to 1). Random text is applied ONLY
    # to `random_langs` (e.g. Telugu & Sanskrit); every other language stays 100%
    # natural. The split is done per exact target length so random and natural
    # rows share the identical language/bucket/length distribution.
    if text_type_pcts is None:
        text_type_pcts = {"natural": 1.0, "random": 0.0}
    nat_pct = text_type_pcts.get("natural", 0.0)
    rand_pct = text_type_pcts.get("random", 0.0)

    plan_lang: List[str] = []
    plan_len: List[int] = []
    plan_source: List[str] = []
    for lang, lcount in zip(langs, lang_counts):
        bucket_counts = apportion(lcount, [length_dist[r] for r in ranges])
        for rng, bcount in zip(ranges, bucket_counts):
            lens = bucket_lengths(rng)
            per_len = apportion(bcount, [1.0] * len(lens))
            for L, c in zip(lens, per_len):
                if lang in random_langs and c > 0:
                    n_nat, n_rand = apportion(c, [nat_pct, rand_pct])
                else:
                    n_nat, n_rand = c, 0
                plan_lang.extend([lang] * (n_nat + n_rand))
                plan_len.extend([L] * (n_nat + n_rand))
                plan_source.extend(["natural"] * n_nat + ["random"] * n_rand)

    plan = Dataset.from_dict({"language": plan_lang, "target_len": plan_len,
                              "source": plan_source})
    return plan.shuffle(seed=seed)


# module-global corpora + config, shared with forked map workers (copy-on-write)
_CORPORA: Dict[str, Dict] = {}
_RANDGEN: Dict[str, "RandomTextGenerator"] = {}
_CFG: Dict = {}


def _gen_batch(batch, indices, rank):
    rank = 0 if rank is None else rank
    rnd = random.Random(hash((_CFG["seed"], rank, indices[0])) & 0xFFFFFFFF)
    abrupt_per = _CFG["abrupt_per"]
    frag_ratio = _CFG["frag_ratio"]
    texts, tlens, frags, cuts, srcs = [], [], [], [], []
    for lang, T, source in zip(batch["language"], batch["target_len"], batch["source"]):
        if source == "random":
            # Synthetic random-grapheme line of exactly `T` graphemes. Use the
            # per-batch `rnd` (uniquely seeded per rank/batch) so forked map
            # workers never duplicate each other's random sequences.
            text = _RANDGEN[lang].generate(target=T, rng=rnd)
            is_frag, cut = False, "none"
        else:
            text, is_frag, cut = generate_one(_CORPORA[lang], T, rnd,
                                               abrupt_per, frag_ratio)
        texts.append(text)
        tlens.append(glen(text))
        frags.append(is_frag)
        cuts.append(cut)
        srcs.append(source)
    return {"text": texts, "text_len": tlens, "is_fragment": frags,
            "cut_type": cuts, "text_source": srcs}


def generate_dataset(plan: Dataset, corpora: Dict[str, Dict],
                     random_generators: Dict[str, "RandomTextGenerator"],
                     abrupt_per: float,
                     frag_ratio: float, num_proc: int = 1, seed: int = 0,
                     batch_size: int = 2000) -> Dataset:
    global _CORPORA, _RANDGEN, _CFG
    _CORPORA = corpora
    _RANDGEN = random_generators or {}
    _CFG = {"abrupt_per": abrupt_per, "frag_ratio": frag_ratio, "seed": seed}
    out = plan.map(
        _gen_batch,
        batched=True,
        batch_size=batch_size,
        with_indices=True,
        with_rank=True,
        num_proc=num_proc,
        remove_columns=["target_len", "source"],
        desc="generate",
    )
    return out


def run(datasets_by_lang: Dict[str, Dataset], text_col: str, lang_pcts: Dict[str, float],
        length_dist: Dict[str, float], total_num_samples: int, abrupt_per: float = 0.10,
        frag_ratio: float = 0.5, num_proc: int = 1, seed: int = 0,
        clean: bool = True, sanskrit_src: str = "Devanagari",
        sanskrit_transliterate: bool = False,
        max_articles: int | None = None,
        text_type_pcts: Dict[str, float] | None = None,
        random_langs: Tuple[str, ...] = (),
        random_generators: Dict[str, "RandomTextGenerator"] | None = None) -> Dataset:
    import datasets as _hfds
    try:
        _hfds.enable_progress_bars()   # make sure map/parquet bars are visible
    except Exception:
        pass

    langs = list(datasets_by_lang)
    corpora = {}
    for k, lang in enumerate(langs, 1):
        print(f"[preprocess {k}/{len(langs)}] {lang} ...", flush=True)
        corpora[lang] = preprocess_corpus(
            datasets_by_lang[lang], text_col, lang, num_proc=num_proc, seed=seed,
            clean=clean, sanskrit_src=sanskrit_src,
            sanskrit_transliterate=sanskrit_transliterate, max_articles=max_articles)

    print(f"[plan] apportioning {total_num_samples:,} samples ...", flush=True)
    plan = make_plan(total_num_samples, lang_pcts, length_dist,
                     text_type_pcts=text_type_pcts, random_langs=random_langs,
                     seed=seed)

    print(f"[generate] producing {len(plan):,} lines "
          f"(num_proc={num_proc}) ...", flush=True)
    out = generate_dataset(plan, corpora, random_generators or {}, abrupt_per,
                           frag_ratio, num_proc=num_proc, seed=seed)
    print("[done] generation complete.", flush=True)
    return out


In [5]:
# --------------------------------------------------------------------------------------
# Random-text generator: builds lines from the grapheme set (uniform, unweighted draws).
# Used for the SYNTHETIC "random" portion of Telugu / Sanskrit lines. The grapheme set,
# word-length distribution and RNG are initialized once and reused for every line.
# --------------------------------------------------------------------------------------
# Realistic word length (in graphemes) distribution: most words are 2-6 aksharas, with a
# short tail of longer ones. Weights are relative.
WORD_LEN_WEIGHTS = {
    1: 3,
    2: 10,
    3: 18,
    4: 20,
    5: 17,
    6: 13,
    7: 9,
    8: 5,
    9: 3,
    10: 2,
}


class RandomTextGenerator:
    """Generate random text lines built from a grapheme set.

    The grapheme set, word-length distribution and RNG (seed) are initialized once
    in the constructor and reused across many calls to `generate`. Each grapheme is
    an independent uniform draw over the full grapheme set (NOT frequency-weighted);
    over many lines every grapheme appears by chance (coupon-collector).

    Args:
        grapheme_dist: dict mapping grapheme (akshara) -> frequency count. Used to
            filter graphemes by `thresh`; beyond that the frequencies are ignored
            (sampling is uniform) and whitespace-only graphemes are skipped (words
            are space-joined here).
        thresh: keep only graphemes whose frequency is >= thresh.
        min_char / max_char: default line length in GRAPHEMES (spaces counted), drawn
            uniformly from [min_char, max_char] when `generate` is called without an
            explicit target.
        seed: optional int for a reproducible RNG (seeded once, at construction).
    """

    def __init__(self, grapheme_dist, thresh=10, min_char=20, max_char=120,
                 seed=None):
        self.rng = random.Random(seed)

        # Keep graphemes that clear the frequency threshold and aren't whitespace.
        self.graphemes = [
            g for g, count in grapheme_dist.items()
            if count >= thresh and g.strip() != ""
        ]
        if not self.graphemes:
            raise ValueError(
                f"no non-whitespace graphemes with frequency >= {thresh}")

        self.min_char = min_char
        self.max_char = max_char
        self.word_lengths = list(WORD_LEN_WEIGHTS)
        self.word_len_weights = list(WORD_LEN_WEIGHTS.values())

    def generate(self, target=None, rng=None):
        """Build and return one random line.

        target: exact line length in graphemes; if None, drawn uniformly from
            [min_char, max_char].
        rng: optional random.Random to sample with (defaults to the instance RNG);
            pass a per-worker RNG to avoid duplicated sequences across processes.
        """
        r = rng if rng is not None else self.rng
        target = r.randint(self.min_char, self.max_char) if target is None else target
        words = []
        n_char = 0  # graphemes accumulated so far (spaces count as 1 each)

        while n_char < target:
            space = 1 if words else 0  # leading space before every word but the first
            remaining = target - n_char - space
            if remaining <= 0:
                break

            # Realistic word length in graphemes, capped so the line lands
            # exactly on `target` graphemes.
            word_len = r.choices(self.word_lengths, weights=self.word_len_weights)[0]
            word_len = min(word_len, remaining)

            # Each grapheme: independent uniform draw over the full set.
            word = "".join(r.choice(self.graphemes) for _ in range(word_len))
            words.append(word)
            n_char += space + word_len

        return " ".join(words)

    def generate_many(self, num_sentences):
        """Return a list of `num_sentences` random lines (uses the instance RNG)."""
        return [self.generate() for _ in range(num_sentences)]

In [6]:

import os

# On Kaggle, point the datasets cache at a big writable disk so the tokenized Arrow
# cache does not fill the small root filesystem (set BEFORE importing/using datasets
# in your notebook; shown here for reference):
#   os.environ["HF_DATASETS_CACHE"] = "/kaggle/temp/hf_cache"


text_col = "text"                     # source text column (configurable)

lang_pcts = {"telugu": 0.70, "sanskrit": 0.15, "english": 0.15}

length_dist = {                       # bucket -> fraction (boundaries fixed, no overlap)
    "1-8":   0.10,
    "9-20":  0.25,
    "21-40": 0.35,
    "41-70": 0.20,
    "71-120": 0.10,
}

abrupt_per = 0.10                     # 5% begin-cut + 5% end-cut
frag_ratio = 0.50                     # short-line frag vs exact-atom split
total_num_samples = 2_500_000
# num_proc = min(8, os.cpu_count() or 1)
num_proc = 5
print(f"Using num_proc of {num_proc}")
seed = 42
# Sanskrit corpus is pre-transcribed to Telugu script, so it is cleaned like Telugu
# (filter + NFKC). Set sanskrit_transliterate=True (+ sanskrit_src) only for raw Devanagari.
sanskrit_transliterate = False
sanskrit_src = "Devanagari"
# For a fast smoke test on Kaggle, cap the corpus, e.g. max_articles = 5000.
max_articles = None

# --------------------------------------------------------------------------------------
# Random-text mix: fraction of each Telugu / Sanskrit line set that is SYNTHETIC random
# graphemes vs NATURAL corpus text. The two fractions must sum to 1. English is always
# natural (it is excluded from `random_langs`).
# --------------------------------------------------------------------------------------
text_type_pcts = {"natural": 0.85, "random": 0.15}   # must sum to 1.0
assert abs(sum(text_type_pcts.values()) - 1.0) < 1e-9, "text_type_pcts must sum to 1"

random_langs = ("telugu", "sanskrit")   # languages that also get random text
grapheme_thresh = 10                     # drop graphemes with corpus frequency < thresh

# Grapheme frequency distributions (grapheme -> count) used to build the random-text
# generators. Provide these JSON files on the run host (e.g. upload as a Kaggle dataset)
# and point these paths at them.
grapheme_dist_paths = {
    "telugu":   "/Users/xai/Personal/Projects/TeluguOCR/src/text_decoder/grapheme_tokenizer/token_dist/telugu_grapheme_dist.json",
    "sanskrit": "/Users/xai/Personal/Projects/TeluguOCR/src/text_decoder/grapheme_tokenizer/token_dist/sanskrit_grapheme_dist.json",
}

# Build one RandomTextGenerator per random language (grapheme set + RNG init'd once).
random_generators = {}
for _lang in random_langs:
    with open(grapheme_dist_paths[_lang]) as _f:
        _dist = json.load(_f)
    random_generators[_lang] = RandomTextGenerator(_dist, thresh=grapheme_thresh,
                                                   seed=seed)
    print(f"[random-text] {_lang}: "
          f"{len(random_generators[_lang].graphemes)} graphemes")

tel_ds = load_dataset(
    "ai4bharat/sangraha",
    data_files = [
        "verified/tel/data-10.parquet",
        "verified/tel/data-11.parquet",
        "verified/tel/data-12.parquet",
        "verified/tel/data-13.parquet",
        "verified/tel/data-14.parquet",
        "verified/tel/data-15.parquet",
        "verified/tel/data-16.parquet",
        "verified/tel/data-17.parquet",
        "verified/tel/data-18.parquet",
        "verified/tel/data-19.parquet",
        "verified/tel/data-20.parquet",
        "verified/tel/data-21.parquet",
        "verified/tel/data-22.parquet",
        "verified/tel/data-23.parquet",
        "verified/tel/data-24.parquet",
        "verified/tel/data-25.parquet",
    ],
    columns=['text']
)['train']


san_ds = load_dataset(
    "harsha-desaraju/sanskrit-text-telugu-script",
    columns=['text']
)['train']


eng_ds = load_dataset(
    "ai4bharat/sangraha",
    data_files = [
        "verified/eng/data-10.parquet",
        "verified/eng/data-11.parquet",
        "verified/eng/data-12.parquet",
        "verified/eng/data-13.parquet",
        "verified/eng/data-14.parquet",
    ],
    columns=['text']
)['train']


datasets_by_lang = {
    "telugu":   tel_ds,
    "sanskrit": san_ds,
    "english":  eng_ds,
}


Using num_proc of 5
[random-text] telugu: 19492 graphemes
[random-text] sanskrit: 13485 graphemes


In [7]:
result = run(datasets_by_lang, text_col, lang_pcts, length_dist,
             total_num_samples, abrupt_per, frag_ratio, num_proc, seed,
             sanskrit_src=sanskrit_src, sanskrit_transliterate=sanskrit_transliterate,
             max_articles=max_articles,
             text_type_pcts=text_type_pcts, random_langs=random_langs,
             random_generators=random_generators)

result.to_parquet("ocr_text_lines.parquet")
print(result)

[preprocess 1/3] telugu ...


index[telugu]: 100%|██████████| 1374/1374 [07:33<00:00,  3.03batch/s]


[preprocess 2/3] sanskrit ...


index[sanskrit]: 100%|██████████| 207/207 [01:59<00:00,  1.73batch/s]

[preprocess 3/3] english ...


...

[generate] producing 2,500,000 lines (num_proc=5) ...


generate (num_proc=5): 100%|██████████| 2500000/2500000 [04:32<00:00, 9184.04 examples/s] 

[done] generation complete.



Creating parquet from Arrow format: 100%|██████████| 6/6 [00:01<00:00,  4.98ba/s]

Dataset({
    features: ['language', 'text', 'text_len', 'is_fragment', 'cut_type', 'text_source'],
    num_rows: 2500000
})


In [8]:
from collections import Counter

Counter(list(result['language']))

Counter({'telugu': 1750000, 'sanskrit': 375000, 'english': 375000})

In [9]:
result.push_to_hub("harsha-desaraju/sample-dataset")

Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00,  5.13ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   0%|          |  524kB /  111MB,  218kB/s  
Processing Files (0 / 1):   1%|          | 1.05MB /  111MB,  328kB/s  
Processing Files (0 / 1):   1%|▏         | 1.57MB /  111MB,  437kB/s  
Processing Files (0 / 1):   2%|▏         | 2.62MB /  111MB,  690kB/s  
Processing Files (0 / 1):   3%|▎         | 3.15MB /  111MB,  786kB/s  
Processing Files (0 / 1):   3%|▎         | 3.67MB /  111MB,  798kB/s  
Processing Files (0 / 1):   6%|▌         | 6.29MB /  111MB, 1.31MB/s  
Processing Files (0 / 1):   7%|▋         | 7.34MB /  111MB, 1.36MB/s  
Processing Files (0 / 1):   7%|▋         | 7.86MB /  111MB, 1.40MB/s  
Processing Files (0 / 1):   8%|▊         | 8.91MB /  111MB, 1.44MB/s  
Processing Files (0 / 1):   8%|▊         | 9.44MB /  111MB, 1.47MB/s  
Processing Files (0 / 1):   9%|▉         | 9.96MB /  111MB, 1.51MB/s  

CommitInfo(commit_url='https://huggingface.co/datasets/harsha-desaraju/sample-dataset/commit/3270a5dbe1055e0aee5ad354e71dc0f82ad667d7', commit_message='Upload dataset', commit_description='', oid='3270a5dbe1055e0aee5ad354e71dc0f82ad667d7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/harsha-desaraju/sample-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='harsha-desaraju/sample-dataset'), pr_revision=None, pr_num=None)

In [10]:
# ======================================================================================
# Image generation + preprocessing setup (fonts, sizes, encoder preprocessing params)
# ======================================================================================
from pathlib import Path

# Fonts uploaded as a Kaggle dataset (same layout as image-generation.ipynb).
FONTS_DIR = "/Users/xai/Personal/Projects/TeluguOCR/data_curation/text_line_images/fonts"

rejected_fonts = ['Pothana2000', 'ponnala', 'Lohit_Telugu', 'Vemana']
danda_non_support_fonts = ["Manu_Bold", "hind-guntur", "Deva_Normal",
                           "AkayaTelivigala-Regular", "AnekTelugu_Condensed-Regular",
                           "Menaka-Italic", "Amruta-Bold", "Sitara-Bold-Italic", "Akshar"]

FONT_SIZES = [30, 32, 34, 36, 38, 40]

font_names = [f.stem for f in Path(FONTS_DIR).rglob("*.ttf")]
font_names = list(set(font_names).difference(set(rejected_fonts)))
# Sanskrit lines use dandas; drop fonts that cannot render them.
sanskrit_fonts = list(set(font_names).difference(set(danda_non_support_fonts)))

# Reserve NUM_VAL_FONTS fonts EXCLUSIVELY for the validation set (unseen at training
# time). They are sampled from the danda-capable set so validation works for every
# language, including Sanskrit. Set `val_fonts` explicitly to pin specific font stems;
# leave it empty to auto-select a fixed (seeded) sample.
NUM_VAL_FONTS = 5
val_fonts = []   # e.g. ["Gautami", "NTR-Regular", ...]; empty -> auto-select
if not val_fonts:
    assert len(sanskrit_fonts) > NUM_VAL_FONTS, "not enough fonts to reserve for validation"
    val_fonts = random.Random(seed).sample(sorted(sanskrit_fonts), NUM_VAL_FONTS)

# Training fonts = everything except the reserved validation fonts.
_val_set = set(val_fonts)
train_fonts = [f for f in font_names if f not in _val_set]
train_sanskrit_fonts = [f for f in sanskrit_fonts if f not in _val_set]

# Encoder image-preprocessing params (see CLAUDE.md): grayscale line images of
# height 64, patch size 8, variable width up to 1024.
image_height = 64
patch_size = 8
max_image_width = 1024

print(f"{len(font_names)} fonts total | train: {len(train_fonts)} "
      f"({len(train_sanskrit_fonts)} sanskrit-capable) | "
      f"val (reserved, unseen): {val_fonts}")

30 fonts total | train: 25 (16 sanskrit-capable) | val (reserved, unseen): ['timmana', 'Gautami-Bold', 'Chathura-Regular', 'NotoSansTelugu_Condensed-Regular', 'NTR-Regular']


In [11]:
# --------------------------------------------------------------------------------------
# Render one line of text to an auto-sized PIL image (from image-generation.ipynb).
# --------------------------------------------------------------------------------------
from PIL import Image, ImageDraw, ImageFont


def generate_image(text, font_path, font_size: int = 40, margin: int = 5,
                   background="white", text_color="black"):
    primary_font = ImageFont.truetype(font_path, font_size)

    # Temporary image just for measuring text
    dummy_img = Image.new("RGB", (1, 1))
    dummy_draw = ImageDraw.Draw(dummy_img)

    # Measure text
    left, top, right, bottom = dummy_draw.multiline_textbbox(
        (0, 0), text, font=primary_font, spacing=4)

    text_width = right - left
    text_height = bottom - top

    # Image size = text size + margins
    width = text_width + 2 * margin
    height = text_height + 2 * margin

    img = Image.new("RGB", (width, height), background)
    draw = ImageDraw.Draw(img)
    draw.text((margin - left, margin - top), text, font=primary_font, fill=text_color)
    return img

In [12]:
# --------------------------------------------------------------------------------------
# Image preprocessing for the encoder (from src/image_encoder/utils.py), WITHOUT the
# final tensor conversion: grayscale -> resize to `image_height` (aspect-preserving,
# capped at `max_image_width`) -> pad width to the nearest multiple of `patch_size`
# (fill=255). Returns a PIL.Image; the ToTensor + Normalize step is intentionally omitted.
# --------------------------------------------------------------------------------------
from torchvision import transforms


class ImagePreprocessor:
    """Preprocess a line image before encoding (no tensor conversion):
    1) convert to grayscale
    2) resize to `image_height`, preserving aspect ratio (capped at `max_image_width`)
    3) pad to the nearest multiple of patch size (fill=255)."""

    def __init__(self, image_height: int, max_image_width: int, patch_size: int):
        assert image_height % patch_size == 0, "Image height should be a multiple of patch size"
        self.image_height = image_height
        self.max_image_width = max_image_width
        self.patch_size = patch_size

    def _transform(self, img: Image.Image) -> Image.Image:
        # Convert to GrayScale
        img = img.convert('L')

        # Calculate the resize target for the image while preserving the aspect ratio
        img_w, img_h = img.size
        scale_factor = self.image_height / img_h
        if scale_factor * img_w > self.max_image_width:
            scale_factor = self.max_image_width / img_w
            target_size = (int(scale_factor * img_h), self.max_image_width)
            diff = self.image_height - target_size[0]
            pad_t, pad_b = diff // 2, diff - diff // 2
            pad_l, pad_r = 0, 0
        else:
            target_size = (self.image_height, int(scale_factor * img_w))
            # Find the nearest multiple of patch size for padding
            diff = (-target_size[1]) % self.patch_size
            pad_l, pad_r = 0, diff
            pad_t, pad_b = 0, 0

        img = transforms.Resize(target_size)(img)
        img = transforms.Pad((pad_l, pad_t, pad_r, pad_b), fill=255)(img)

        return img

    def __call__(self, img: Image.Image) -> Image.Image:
        return self._transform(img)

In [1]:
# --------------------------------------------------------------------------------------
# For every text line: render it with a random (language-appropriate) font + size, then
# preprocess to the encoder's grayscale height-64 format. The preprocessed PIL image
# overwrites the `image` column; `font` / `font_size` are kept as metadata.
# --------------------------------------------------------------------------------------
preprocessor = ImagePreprocessor(image_height, max_image_width, patch_size)


def create_image_for_sample(sample, font_pool, sanskrit_pool):
    if sample['language'] == 'sanskrit':
        chosen_font = random.choice(sanskrit_pool)
    else:
        chosen_font = random.choice(font_pool)
    font_path = f"{FONTS_DIR}/{chosen_font}.ttf"
    chosen_font_size = random.choice(FONT_SIZES)

    img = generate_image(sample['text'], font_path, chosen_font_size)
    sample['image'] = preprocessor(img)      # grayscale, height 64, padded; no tensor
    sample['font'] = chosen_font
    sample['font_size'] = chosen_font_size
    return sample


# Hold out VAL_SIZE lines for validation; the rest are training. Validation images are
# rendered ONLY with the reserved fonts (unseen during training); training images use
# the remaining fonts. Font pools are passed per-split via fn_kwargs.
VAL_SIZE = 5_000
_split = result.train_test_split(test_size=VAL_SIZE, seed=seed, shuffle=True)

train_ds = _split["train"].map(
    create_image_for_sample, num_proc=num_proc,
    fn_kwargs={"font_pool": train_fonts, "sanskrit_pool": train_sanskrit_fonts},
)
val_ds = _split["test"].map(
    create_image_for_sample, num_proc=num_proc,
    fn_kwargs={"font_pool": val_fonts, "sanskrit_pool": val_fonts},
)

dataset = DatasetDict({"train": train_ds, "validation": val_ds})
dataset

NameError: name 'ImagePreprocessor' is not defined

In [ ]:
dataset.push_to_hub(
    "harsha-desaraju/synthetic-line-text-images"
)